# Fine-Tuned Filtering and Registration

This notebook is a more configurable variant of the filtering and registration workflow. It shows how to tune filter settings globally, per channel, and, where supported, per time point.

Author: Fabrizio Musacchio
Date: June 2026

Run the notebook from top to bottom for the packaged example data. To adapt it to your own microscopy data, change the input path and selected channels first, then tune method-specific parameters as described in the Markdown cells.

Several cells open napari viewers. If you run on a headless system, skip those visualization cells and keep the processing and saving cells.


## Imports

Import the package functions used below and locate the repository root. In notebook form, the project root is discovered from the current working directory so the notebook can be opened either from the repository root or from this `notebooks` folder.


In [ ]:
from __future__ import annotations

from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "spectral_unmixing").exists() and (candidate / "example_data").exists():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("Could not find the spectral-unmixing project root. "
                       "Start this notebook from inside the repository.")

from spectral_unmixing.filters import apply_filters, match_histograms_across_time, max_z_project
from spectral_unmixing.io import load_stack_with_omio, write_stack_with_omio
from spectral_unmixing.registration import correct_intra_stack_z_drift, register_stack

import omio as om


## Input And Output Paths

Choose the input example data and define where results will be written. To use your own data, this is usually the only cell where you need to change paths.


In [ ]:
INPUT_PATH = (PROJECT_ROOT / "example_data" / "Gockel_Nieves_Rivera_2026" / "Gockel_Nieves_Rivera_2026_5D_stack.tif")
INPUT_NAME = INPUT_PATH.stem

OUTPUT_DIR = INPUT_PATH.parent / "registered_fine_filtered"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = OUTPUT_DIR / f"{INPUT_NAME}_registered_fine_filtered.tif"


## User Settings

Collect high-level registration and projection settings in one place. Change these values before running the downstream processing cells.


In [ ]:
REGISTRATION_CHANNEL = 0
INTRA_STACK_METHOD = "pystackreg"
TEMPORAL_REGISTRATION_METHOD = "pystackreg"
NEIGHBOR_WINDOW_SIZE = 3
PROJECTION_ZRANGE = (10, 19)


## Filter Settings

Configure the filter sequence and filter strengths. In the fine-tuning example, filter sizes and Gaussian sigmas can be scalar values or time-point-specific lists.


In [ ]:
# Base filter settings applied to all channels unless overridden for channel 2
# (the second channel, index 1).
FILTERS = ["median", "gaussian"]
FILTERS_CHANNEL2 = ["median"]

# Per-time-point filtering: when list length matches T, each time point gets its
# own value. Otherwise only the first entry is used for all time points.
MEDIAN_SIZE = [3, 3, 3, 3, 3, 3, 3, 3, 3]
GAUSSIAN_SIGMA = [0.8, 0.8, 0.9, 1.0, 1.0, 0.9, 0.8, 0.8, 0.8]

# Optional second-channel-specific overrides.
MEDIAN_SIZE_CHANNEL2 = [5, 5, 5, 5, 5, 5, 5, 5, 5]
GAUSSIAN_SIGMA_CHANNEL2 = None

POST_FILTERS = ["median", "gaussian"]
POST_FILTERS_CHANNEL2 = ["median"]
POST_MEDIAN_SIZE = 3
POST_GAUSSIAN_SIGMA = 1.2
POST_MEDIAN_SIZE_CHANNEL2 = 5
POST_GAUSSIAN_SIGMA_CHANNEL2 = None


## Load Stack With OMIO

Load the stack through OMIO via the package I/O helper. The returned array is expected in canonical `TZCYX` order, which keeps time, z, channel, y, and x indexing explicit.


In [ ]:
stack, metadata = load_stack_with_omio(INPUT_PATH)
print(f"Loaded stack: {stack.shape}, axes={metadata.get('axes')}")
om.open_in_napari(stack, metadata, "Unmixed raw |")


## Correct Intra-Stack Z-Drift

Correct slice-to-slice motion inside each 3D time point. The registration is estimated from the selected stable channel and then applied to the full multichannel stack.


In [ ]:
z_corrected_stack = correct_intra_stack_z_drift(
    stack,
    registration_channel=REGISTRATION_CHANNEL,
    method=INTRA_STACK_METHOD,
    reference_mode="neighbor",
    neighbor_window_size=NEIGHBOR_WINDOW_SIZE,
    pre_median_filter=True,
    post_median_filter=False,
    median_kernel_size=3,
    verbose=True)
print(f"Z-corrected stack: {z_corrected_stack.shape}")
z_corrected_metadata = om.update_metadata_from_image(metadata, z_corrected_stack)
om.open_in_napari(z_corrected_stack, z_corrected_metadata, "Z-corrected |")


## Register Stack Across Time

Register the time series to the first time point. Registration shifts are estimated from a max-z projection of the selected channel and then applied to the original stack.


In [ ]:
registered_stack = register_stack(
    z_corrected_stack,
    registration_channel=REGISTRATION_CHANNEL,
    method=TEMPORAL_REGISTRATION_METHOD,
    zrange=PROJECTION_ZRANGE,
    pre_median_filter=True,
    post_median_filter=True,
    median_kernel_size=5,
    verbose=True,)
print(f"Registered stack: {registered_stack.shape}")
registered_metadata = om.update_metadata_from_image(metadata, registered_stack)
om.open_in_napari(registered_stack, registered_metadata, "Registered |")


## Match Histograms Across Time

Match time-point intensity distributions to the reference time point. This can reduce slow brightness changes before filtering and projection.


In [ ]:
matched_stack = match_histograms_across_time(registered_stack, reference_t=0)
print(f"Histogram matched stack: {matched_stack.shape}")
matched_metadata = om.update_metadata_from_image(metadata, matched_stack)
om.open_in_napari(matched_stack, matched_metadata, "Registered + hist matched |")


## Filter Registered Stack

Apply the configured denoising filters to the registered stack while keeping the canonical `TZCYX` structure intact.


In [ ]:
filtered_stack = apply_filters(
    matched_stack,
    filters=FILTERS,
    filters_channel2=FILTERS_CHANNEL2,
    median_size=MEDIAN_SIZE,
    gaussian_sigma=GAUSSIAN_SIGMA,
    median_size_channel2=MEDIAN_SIZE_CHANNEL2,
    gaussian_sigma_channel2=GAUSSIAN_SIGMA_CHANNEL2,
    apply_3d=False)
print(f"Filtered stack: {filtered_stack.shape}")
filtered_metadata = om.update_metadata_from_image(metadata, filtered_stack)
om.open_in_napari(filtered_stack, filtered_metadata, "Fine filtered |")


## Max-Z-Project

Collapse the z-axis by maximum-intensity projection while preserving time and channel axes. Use `zrange` to restrict the projected z-slices if needed.


In [ ]:
projected_stack = max_z_project(filtered_stack, zrange=PROJECTION_ZRANGE)
print(f"Projected stack: {projected_stack.shape}")
projected_metadata = om.update_metadata_from_image(metadata, projected_stack)
om.open_in_napari(projected_stack, projected_metadata, "Projected |")


## Filter Projected Stack Again

Apply a second, usually gentler, filter pass after projection. This can improve visualization of projected data but should be tuned conservatively.


In [ ]:
filtered_projected_stack = apply_filters(
    projected_stack,
    filters=POST_FILTERS,
    filters_channel2=POST_FILTERS_CHANNEL2,
    median_size=POST_MEDIAN_SIZE,
    gaussian_sigma=POST_GAUSSIAN_SIGMA,
    median_size_channel2=POST_MEDIAN_SIZE_CHANNEL2,
    gaussian_sigma_channel2=POST_GAUSSIAN_SIGMA_CHANNEL2,
    apply_3d=False)
print(f"Filtered projected stack: {filtered_projected_stack.shape}")
filtered_projected_metadata = om.update_metadata_from_image(metadata, filtered_projected_stack)
om.open_in_napari(filtered_projected_stack, filtered_projected_metadata, "Fine filtered projected |")


## Save Filtered Projected Stack With OMIO

Save the processed stack through OMIO. The input file is not overwritten; results are written to the configured output folder.


In [ ]:
saved_output = write_stack_with_omio( OUTPUT_PATH, filtered_projected_stack, metadata,)
print(saved_output)


## End

The notebook is complete. Saved outputs can be reopened from the output folder or reused in downstream analysis scripts.
